# Fintech NLP Microservices Prototype
This notebook demonstrates three core fintech services using the Groq API:
1. **Earnings Summarizer**: Condensing earnings call snippets with ROUGE evaluation.
2. **Support Ticket Classifier**: 5-class classification with Chain-of-Thought (CoT) reasoning.

In [ ]:
!pip install -q langchain langchain-groq rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.6 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from rouge_score import rouge_scorer

# Initialize Groq LLM
# Ensure GROQ_API_KEY is in your Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.1)

## 1. Earnings Summarizer
First, we generate synthetic earnings call snippets, then summarize them using Zero-Shot and Few-Shot techniques, and finally evaluate the results.

In [ ]:
# a) Generate Earning Call Snippets
gen_query = "Generate 3 short, realistic earnings call snippets for a fintech company regarding revenue, user growth, and regulatory challenges."
snippets_resp = llm.invoke([("user", gen_query)])
snippets = [s.strip() for s in snippets_resp.content.split('\n\n') if s.strip()]
print("--- Generated Snippets ---\n", snippets_resp.content)

--- Generated Snippets ---
 Here are three short, realistic earnings call snippets for a fintech company:

**Snippet 1: Revenue Growth**
"We're pleased to report that our revenue for the quarter came in at $23.4 million, representing a 32% increase year-over-year. This growth was driven primarily by the expansion of our payment processing services, which saw a 45% increase in transaction volume. Additionally, our subscription-based models continue to gain traction, with a 25% increase in monthly recurring revenue. We're confident that our diversified revenue streams will continue to drive growth and profitability for the company."

**Snippet 2: User Growth and Engagement**
"Our user base continues to grow at a rapid pace, with a 50% increase in active users over the past 12 months. We now have over 5 million registered users on our platform, with an average monthly retention rate of 75%. We're also seeing strong engagement metrics, with the average user logging in 3-4 times per week an

In [ ]:
# b) Summarization Logic
def get_summary(text, mode="zero-shot"):
    if mode == "zero-shot":
        prompt = f"Summarize the following earnings call snippet in one concise sentence:\n\nSnippet: {text}\nSummary:"
    else:
        prompt = f"""Summarize the snippet into one concise sentence.\n\nExample 1:\nSnippet: We saw a 20% increase in net profit due to lower operational costs.\nSummary: Net profit rose 20% driven by cost efficiencies.\n\nExample 2:\nSnippet: New regulatory requirements in the EU might slow down our expansion plans.\nSummary: Potential EU regulatory hurdles may delay expansion.\n\nSnippet: {text}\nSummary:"""

    res = llm.invoke([("user", prompt)])
    return res.content

# Summarize the first snippet
sample_text = snippets[0] if snippets else "Default text"
zero_shot_summary = get_summary(sample_text, mode="zero-shot")
few_shot_summary = get_summary(sample_text, mode="few-shot")

print(f"Zero-Shot: {zero_shot_summary}")
print(f"Few-Shot: {few_shot_summary}")

Zero-Shot: There is no earnings call snippet provided to summarize.
Few-Shot: The provided snippet contains three hypothetical earnings call excerpts for a fintech company, but no actual snippet is given to summarize.


In [ ]:
# c) Evaluate with ROUGE-L
# Since we don't have a human reference, we'll treat Few-Shot as the target for this demo
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
scores = scorer.score(few_shot_summary, zero_shot_summary)
print(f"ROUGE-L Score between Zero-Shot and Few-Shot: {scores['rougeL']}")

ROUGE-L Score between Zero-Shot and Few-Shot: Score(precision=0.5555555555555556, recall=0.23809523809523808, fmeasure=0.33333333333333326)


## 2. Support Ticket Classifier (CoT)
Classifies tickets into: Billing, Tech, Refund, General, or Escalate.

In [ ]:
# Support Ticket Classifier
ticket_text = "I was charged twice for my premium subscription and I want my money back immediately!"

classifier_prompt = f"""Classify the following customer support ticket into one of these 5 classes:
(Billing, Tech, Refund, General, Escalate).

Provide your reasoning step-by-step, then state the final class.

Ticket: {ticket_text}

Reasoning:"""

response = llm.invoke([("user", classifier_prompt)])
print(response.content)

To classify the customer support ticket, let's break it down step by step:

1. **Identify the main issue**: The customer is complaining about being charged twice for their premium subscription. This indicates a problem related to payment or transaction.

2. **Determine the customer's goal**: The customer wants their money back immediately, which suggests they are seeking a refund or correction of the billing error.

3. **Consider the classes**:
   - **Billing**: This class involves issues related to payments, invoices, or subscription charges. The customer's complaint about being charged twice fits into this category.
   - **Tech**: This class typically involves technical issues with a product or service, such as bugs, functionality problems, or compatibility issues. The customer's issue does not seem to be related to the technical aspect of the premium subscription.
   - **Refund**: This class is specifically for requests to return money to the customer. While the customer does ask fo